Scikit-learn Logistic Regression Model

In [ ]:
# 运行数据准备脚本：下载 IMDB 数据并切分成 train/validation/test 三个 CSV
# 【bug修复】原为 `!python download-prepare-dataset.py`(连字符),但该目录下实际文件名是
# `download_prepare_dataset.py`(下划线),连字符会报「can't open file」。已改为下划线。
!python download_prepare_dataset.py

In [ ]:
import pandas as pd

# 读入准备好的三份数据
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

Scikit-learn baseline

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
vectorizer = CountVectorizer()  # 词袋(Bag-of-Words)特征：统计每条评论中各词出现次数

# 只在训练集 fit_transform（学习词表+转换），验证/测试集只 transform（复用同一词表，避免数据泄漏）
X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

y_train, y_val, y_test = train_df["label"], val_df["label"], test_df["label"]  # 取标签列
def eval(model, X_train, y_train, X_val, y_val, X_test, y_test):
    # Making predictions（在三个划分上分别预测）
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    # Calculating accuracy and balanced accuracy（计算准确率；balanced 相关行为原作者注释掉的备选指标）
    accuracy_train = accuracy_score(y_train, y_pred_train)
    # balanced_accuracy_train = balanced_accuracy_score(y_train, y_pred_train)

    accuracy_val = accuracy_score(y_val, y_pred_val)
    # balanced_accuracy_val = balanced_accuracy_score(y_val, y_pred_val)

    accuracy_test = accuracy_score(y_test, y_pred_test)
    # balanced_accuracy_test = balanced_accuracy_score(y_test, y_pred_test)

    # Printing the results
    print(f"Training Accuracy: {accuracy_train*100:.2f}%")
    print(f"Validation Accuracy: {accuracy_val*100:.2f}%")
    print(f"Test Accuracy: {accuracy_test*100:.2f}%")
from sklearn.dummy import DummyClassifier

# Create a dummy classifier with the strategy to predict the most frequent class
# 哑分类器：永远预测训练集中最常见的类别，作为衡量模型是否真学到东西的最低参照线
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

eval(dummy_clf, X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
# 逻辑回归：词袋特征上的经典线性分类器；max_iter=1000 保证在高维稀疏特征上收敛
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
eval(model, X_train, y_train, X_val, y_val, X_test, y_test)